# Anima 2B SnapRef-style training
Separate reference rank-128 and target rank-64 LoRAs on frozen Anima base v1.0. Reference caption defaults to empty. Reuses the saved artist/date index and velocity L2 only. Both LoRA branches and the cs (character + style) reference-mode embedding start from the published BF16 SnapRef v1a adapter, then are finetuned with velocity L2.


In [ ]:
import io, json, os, shutil, stat, subprocess, sys, zipfile
from pathlib import Path
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('This notebook is for Google Colab only; do not run it locally.')
from google.colab import files, userdata
print('Colab guard passed. No installer has run yet.')


In [ ]:
# Default: clone or fast-forward the committed fork. Upload/existing remain optional.
SOURCE = 'git'  # 'git', 'upload', or 'existing'
REPO_URL = 'https://github.com/TheDucker1/sd-scripts'
BRANCH = 'style_loss'
REPO = Path('/content/sd-scripts')
if SOURCE == 'upload':
    uploaded = files.upload()
    archives = [data for name, data in uploaded.items() if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise RuntimeError('Upload exactly one source-only repository ZIP.')
    destination = Path('/content/moss-upload')
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(archives[0])) as archive:
        if sum(info.file_size for info in archive.infolist()) > 512 * 1024**2:
            raise RuntimeError('Source ZIP exceeds 512 MiB; exclude models, datasets and environments.')
        for info in archive.infolist():
            target = (destination / info.filename).resolve()
            if destination not in target.parents or stat.S_ISLNK(info.external_attr >> 16):
                raise RuntimeError('Unsafe archive entry: ' + info.filename)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)
    candidates = list(destination.rglob('anima_train_style.py'))
    if len(candidates) != 1:
        raise RuntimeError('Expected exactly one current Anima style trainer in the ZIP.')
    REPO = candidates[0].parent
    del archives, uploaded
elif SOURCE == 'git':
    if not REPO_URL:
        raise RuntimeError('Set REPO_URL.')
    if not REPO.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
    else:
        def git_output(*args):
            return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()
        if git_output('remote', 'get-url', 'origin').rstrip('/').removesuffix('.git') != REPO_URL.rstrip('/').removesuffix('.git'):
            raise RuntimeError('Existing checkout has a different origin.')
        if git_output('symbolic-ref', '--short', 'HEAD') != BRANCH:
            raise RuntimeError('Existing checkout is on a different branch.')
        if git_output('status', '--porcelain', '--untracked-files=no'):
            raise RuntimeError('Tracked files have local edits. Commit or stash them before updating; no edits were discarded.')
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', 'FETCH_HEAD'], check=True)
    print('Source commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
elif SOURCE != 'existing':
    raise ValueError('Unknown SOURCE')
SETUP = REPO / 'tools/colab/setup_moss_style.py'
if not SETUP.is_file():
    raise RuntimeError('Missing setup script; commit and push all new Anima files to the selected fork/branch first.')
print('Using source:', REPO)


In [ ]:
# One-time authentication for this runtime. Never put your token in code or chat.
# Token is inherited by setup/build subprocesses; no persistent HF login.
from getpass import getpass
FORCE_TOKEN_PROMPT = False  # Set True to replace an expired/rejected token.
token = None if FORCE_TOKEN_PROMPT else os.environ.get('HF_TOKEN')
if not token and not FORCE_TOKEN_PROMPT:
    try:
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None
if not token:
    token = getpass('Hugging Face read token (hidden, this runtime only): ').strip()
if not token or not token.strip():
    raise RuntimeError('Provide a Hugging Face read token for model setup.')
os.environ['HF_TOKEN'] = token.strip()
print('HF token is available for this runtime. No token value is printed or written to disk.')
del token


In [ ]:
DOWNLOAD_MODELS = True
TEXT_ENCODER = 'bf16'  # Qwen3-0.6B-Base; no ConvRot profile.
# W4A8/GPTQ, GGUF, and arbitrary replacement encoders are not supported here.
PERSIST_MODELS_TO_DRIVE = False
VENV = Path('/content/moss-venv')
MODELS = Path('/content/anima-models')
if PERSIST_MODELS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    MODELS = Path('/content/drive/MyDrive/anima-models')
print('Environment:', VENV, '\nAssets:', MODELS)


In [ ]:
# This is the only installation/download step. Run it on the Colab VM.
# Setup validates the token before downloading model weights; DINOv3 is no longer needed.
command = [sys.executable, str(SETUP), '--repo', str(REPO), '--repo-url', REPO_URL, '--branch', BRANCH, '--venv', str(VENV), '--models-dir', str(MODELS), '--text-encoder', TEXT_ENCODER]
command += ['--manifest', str(REPO / 'configs/anima_colab_models.json')]
if not DOWNLOAD_MODELS:
    command.append('--skip-models')
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
# Inspect metadata only. These cells do not load checkpoint tensors.
print((VENV / 'moss-runtime.json').read_text())
for name in ['resolved_models.json', 'dataset_stream.json']:
    path = MODELS / name
    if path.exists():
        print(name, '->', path)
print('Use this interpreter for later commands:', VENV / 'bin/python')
print('Preparation only: no training has started.')


Loads Anima base v1.0, Qwen3-0.6B-Base, its T5 tokenizer (no T5 encoder), Qwen Image VAE, PixAI and the BF16 SnapRef v1a adapter. Reference and target adapters start from pretrained SnapRef v1a; its cs mode embedding initializes the learned style mode.


## Stage the offline artist archive in RAM
Upload the completed downloader folder as an **uncompressed `.tar`** to Drive. The folder must include `manifest.json`, `artist-index.npz` and `images/`; a containing folder in the tar is fine. No index rebuild or remote dataset fetching is performed.
The next cell copies the tar from Drive to `/dev/shm` once, with progress and partial-copy resume, then validates its saved image checksums and builds local tar offsets. Training and previews read compressed image bytes directly from this RAM-backed tar, without extracting images or allocating a second full dataset copy.
`DATA_STORAGE='ram'` requires a real tmpfs and checks its capacity plus available RAM, leaving 12 GiB for model loading/training by default. If the whole tar does not fit, preparation stops; choose a smaller dataset or explicitly set `DATA_STORAGE='local'` to use the Colab SSD instead. It never silently falls back to reading images from Drive. Cached data survives training process restarts but not Colab runtime resets.
Both download selection and training enforce the saved same-artist, distinct-image date window (365 days here). Eligible pair counts are unordered; a roster can contain images more than one year apart, but those images will never be paired with each other.


In [ ]:
# Tiny synthetic offline fixtures only: no dataset fetches, pretrained weights, or GPU.
subprocess.run([str(VENV / 'bin/python'), '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_stage_artist_dataset.py', '-v'], cwd=REPO, check=True)
subprocess.run([str(VENV / 'bin/python'), '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_anima_reference.py', '-v'], cwd=REPO, check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ARCHIVE = Path('/content/drive/MyDrive/moss-training/artists-pair-dataset.tar')
DATA_STORAGE = 'ram'  # 'ram': real tmpfs; 'local': Colab SSD, also avoids Drive reads during training.
RESERVE_RAM_GIB = 12  # RAM left available for model loading, training and CPU prefetch.
if DATA_STORAGE not in ('ram', 'local'):
    raise ValueError("DATA_STORAGE must be 'ram' or 'local'")
INDEX_DIR = Path('/dev/shm/anima-artist-dataset' if DATA_STORAGE == 'ram' else '/content/anima-artist-dataset')
if not DATA_ARCHIVE.is_file():
    raise FileNotFoundError(f'Upload the completed dataset .tar to {DATA_ARCHIVE}, or change DATA_ARCHIVE.')
stage_command = [str(VENV / 'bin/python'), '-u', str(REPO / 'tools/colab/stage_artist_dataset.py'),
                 '--archive', str(DATA_ARCHIVE), '--output', str(INDEX_DIR),
                 '--storage', DATA_STORAGE, '--reserve-ram-gib', str(RESERVE_RAM_GIB)]
subprocess.run(stage_command, cwd=REPO, check=True)
index_manifest = json.loads((INDEX_DIR / 'manifest.json').read_text())
assert index_manifest['complete'] and index_manifest['storage'] == 'local_tar'
print('Ready:', index_manifest['image_count'], 'images;', index_manifest['artist_count'], 'artists;', DATA_STORAGE)
print('Pair window:', index_manifest['pair_max_days'], 'days; no image reads from Drive during training.')


## Training settings and persistence
Training uses Anima asymmetric reference attention, velocity L2 only. CHECKPOINT_BLOCKS selects the first N of 28 blocks; 0 disables recomputation. Batch 2 with accumulation 2 gives four pairs per update.

| Setting | Default | Meaning |
|---|---|---|
| PIXEL_BUDGET | 768 Ã— 768 | Maximum area **per image**, for both A and B. Aspect buckets use 64-pixel increments; shapes vary and rounding can leave unused area. |
| MAX_STEPS | 100000 | Total optimizer updates, including updates completed before resume. |
| PREVIEW_EVERY / PREVIEW_PAIRS | 1000 / 5 | Five fresh A/B/result JPEG triptychs, each using a full 50-step inference run. |
| SAVE_EVERY / KEEP_LAST | 1000 / 3 | Keep three completed adapter + resume checkpoints. Preview images and logs remain. |
| LEARNING_RATE / WARMUP_STEPS | 1e-4 / 1000 | AdamW8bit peak LR and initial warmup. |
| LR_CYCLES / MIN_LR_RATIO | 5 / 0.01 | Smooth cosine rises/falls with a linearly decaying envelope, ending at 1e-6. This is a starting preset, not a guarantee of convergence. |

The command uses `--compile --compile_backend inductor --compile_scope blocks --compile_modules dit --compile_dynamic true`: only Anima reference/target block functions compile, with dynamic shapes. Compilation can pause initial progress. Checkpoints exclude frozen model weights; `training_state.pt` stores optimizer, scheduler and RNG. Resume restarts random pair sampling without replaying earlier batches. Artist pairs are sampled from the full RAM index, so this backend does not use the sequential stream's shuffle window. An optional bounded CPU producer prefetches two batches to overlap image decoding and preparation with GPU work. Drive writes can be slow; only completed checkpoints are auto-selected.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_NAME = 'anima-snapref-flow-l2-768-ram'
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 2
PIXEL_BUDGET = 768 * 768
SHUFFLE_BUFFER = 0  # Artist backend samples randomly from the full RAM index.
PREFETCH_BATCHES = 2  # Bounded CPU batch producer, overlapping data loading with GPU work.
MAX_STEPS = 100_000
PREVIEW_EVERY, PREVIEW_PAIRS, PREVIEW_STEPS = 1000, 5, 50
SAVE_EVERY, KEEP_LAST = 1000, 3
LEARNING_RATE, WARMUP_STEPS = 1e-4, 1000
LR_CYCLES, MIN_LR_RATIO = 5, 0.01
COMPILE = True
GRADIENT_CHECKPOINTING = False
CHECKPOINT_BLOCKS = 0  # Overrides GRADIENT_CHECKPOINTING; None uses its all-or-none behavior.
TAG_DROPOUT = 0.15  # Retain about 85% of threshold-selected tags on average.
OFFLOAD_TEXT_ENCODER = False  # Keep Qwen weights and compute on the GPU.
OUTPUT_DIR = Path('/content/drive/MyDrive/moss-training') / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config = json.loads((REPO / 'configs/anima_style_colab.json').read_text())
config['assets_manifest'] = str(MODELS / 'resolved_models.json')
config['offload_text_encoder'] = OFFLOAD_TEXT_ENCODER
config['gradient_checkpointing'] = GRADIENT_CHECKPOINTING
config['checkpoint_blocks'] = CHECKPOINT_BLOCKS
config['content_loss'] = 'flow_l2'
config['data'] = {'artist_index': str(INDEX_DIR), 'index_fetch_workers': 2, 'index_cache_bytes': 0, 'datasets': [{'batch_size': BATCH_SIZE, 'resolution': [768, 768], 'enable_bucket': True, 'max_pixels': PIXEL_BUDGET, 'min_bucket_reso': 256, 'max_bucket_reso': 1536, 'bucket_reso_steps': 64, 'bucket_no_upscale': False, 'max_aspect_ratio': 3.0, 'repeat': True, 'shuffle_buffer': SHUFFLE_BUFFER, 'shuffle_caption': True, 'resize_interpolation': 'area', 'subsets': [{}]}]}
config['data']['datasets'][0]['caption_tag_dropout_rate'] = TAG_DROPOUT
config['training'].update(batch_size=BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION, max_steps=MAX_STEPS, save_every=SAVE_EVERY, keep_last_checkpoints=KEEP_LAST, learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS, lr_cycles=LR_CYCLES, min_lr_ratio=MIN_LR_RATIO, prefetch_batches=PREFETCH_BATCHES)
config['sampling'].update(every_n_steps=PREVIEW_EVERY, num_pairs=PREVIEW_PAIRS, steps=PREVIEW_STEPS)
RUN_CONFIG = OUTPUT_DIR / 'launch_config.json'
if RUN_CONFIG.exists() and json.loads(RUN_CONFIG.read_text()) != config:
    raise RuntimeError('Existing run settings differ. Restore the settings or choose a new RUN_NAME.')
RUN_CONFIG.write_text(json.dumps(config, indent=2) + '\n')
print('Persistent run:', OUTPUT_DIR)
print(json.dumps(config['training'], indent=2))


Run the TensorBoard cell before launching training to keep its dashboard visible while the next cell runs. Losses, learning rate and gradient norm are also written to `metrics.jsonl`; console output goes to `console.log`. JPEG previews are under `samples/step-XXXXXX/` on Drive. Colab runtime limits still apply; this notebook does not keep a disconnected runtime alive.

In [ ]:
TB_LOGDIR = str(OUTPUT_DIR / 'tensorboard')
%load_ext tensorboard
%tensorboard --logdir $TB_LOGDIR


In [ ]:
# Load the downloaded pretrained components and train the editing LoRA.
# Missing/incompatible checkpoints raise an error; there is no random fallback.
import re, shlex
AUTO_RESUME = True
checkpoints = sorted((p for p in OUTPUT_DIR.iterdir() if re.fullmatch(r'step-\d+', p.name) and all((p / name).is_file() for name in ['complete.json', 'adapter_model.safetensors', 'adapter_config.json', 'training_state.pt'])), key=lambda p: int(p.name.split('-')[1]))
command = [str(VENV / 'bin/python'), '-u', str(REPO / 'anima_train_style.py'), '--config', str(RUN_CONFIG), '--output_dir', str(OUTPUT_DIR), '--device', 'cuda']
if COMPILE:
    command += ['--compile', '--compile_backend', 'inductor', '--compile_scope', 'blocks', '--compile_modules', 'dit', '--compile_dynamic', 'true']
if AUTO_RESUME and checkpoints:
    command += ['--resume', str(checkpoints[-1])]
elif checkpoints:
    raise RuntimeError('Existing checkpoints found. Enable AUTO_RESUME or choose a new RUN_NAME.')
print(shlex.join(command), flush=True)
with (OUTPUT_DIR / 'console.log').open('a', buffering=1) as log:
    log.write('\nCOMMAND: ' + shlex.join(command) + '\n')
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
        if process.wait():
            raise RuntimeError('Training failed. Read the output above and console.log on Drive.')
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()
